In [2]:
import gradio as gr
import base64

from dotenv import load_dotenv
from openai import OpenAI
from io import BytesIO
from PIL import Image

load_dotenv()

True

In [3]:
gpt = OpenAI()
llama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [4]:
MODELS = {'llama 3.2': 'llama3.2', 'GPT 5 MINI': 'gpt_5_mini'}
PROMPT_HEADER = 'FINAL PROMPT:'
SYSTEM_PROMPT = f"""
You are an assistant that helps users create high-quality image generation prompts.

Your job:
- Ask clarifying questions step by step
- Do NOT generate the final prompt too early
- Gather details like:
  - subject
  - style
  - composition
  - lighting
  - mood
  - camera / perspective

When you have enough information, output:

{PROMPT_HEADER}
<well-written detailed prompt>

Keep responses short and conversational.
"""


def stream_chat(client, model, message, history):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    

    for record in history:
        messages.append({"role": record["role"], "content": record["content"][0]["text"]})
        
    messages.append({"role": "user", "content": message})

    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )

    partial_reply = ""
    history_chunk = ""
    prompt = ""
    for chunk in stream:
        if chunk.choices[0].delta.content:
            partial_reply += chunk.choices[0].delta.content
            history_chunk = [{"role": "user", "content": [{"text": message, "type": "text"}]},
                            {"role": "assistant", "content": [{"text": partial_reply, "type": "text"}]}]
            _, _, prompt_chunk = partial_reply.partition(PROMPT_HEADER)
            
            yield history + history_chunk, "", (prompt + prompt_chunk).strip()


def msg_submit(message, history, model):
    model_name = MODELS[model]
    if model_name == 'llama3.2':
        yield from stream_chat(llama, model_name, message, history)
    elif model_name == 'gpt_5_mini':
        yield from stream_chat(gpt, model_name, message, history)
    else:
        raise Exception(f'Unknown model name: {model_name}')

In [15]:
def artist(prompt: str):
    if not prompt:
        return None
    response = gpt.images.generate(
        model="gpt-image-1-mini",
        prompt=prompt,
        size="1024x1024",
        n=1,
        # response_format="b64_json",
    )
    image_data = base64.b64decode(response.data[0].b64_json)
    img = Image.open(BytesIO(image_data))
    return img


In [16]:
with gr.Blocks() as demo:
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## Chat")
            model = gr.Dropdown(label='Model', choices=MODELS.keys(), value='llama 3.2')
            chatbot = gr.Chatbot()
            msg = gr.Textbox(value="нарисуй козу")
            clear = gr.Button("Clear chat")

        with gr.Column(scale=1):
            gr.Markdown("## Image Panel")
            prompt_box = gr.Textbox(label="Prompt", lines=5)
            run_btn = gr.Button("Run")
            output_box = gr.Image(label="Result Image")
            

    msg.submit(msg_submit, [msg, chatbot, model], [chatbot, msg, prompt_box])
    clear.click(lambda: [], None, chatbot)

    run_btn.click(artist, prompt_box, output_box)

if __name__ == "__main__":
    demo.launch(inbrowser=False)

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
